In [66]:
import networkx as nx
import random

import pandas as pd
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt

In [67]:
#Get the graph
#graph2 = nx.read_graphml("./graphs/groupsGraph.graphml")

graph = nx.dorogovtsev_goltsev_mendes_graph(7)
#Get the edges and split into G_training and G_test
k = 60
edgesFull = list(graph.edges())
testEdges = random.sample(edgesFull, k) #Take 15% of edges away
trainingEdges = list(set(edgesFull) - set(testEdges))

#Need to add random edges

G_training = graph.copy()
G_training.remove_edges_from(testEdges)
G_testing = graph.copy()
G_testing.remove_edges_from(trainingEdges)

In [68]:
pred = nx.adamic_adar_index(G_training)

#need to sort the list of tuples based on the last value
sortedPred = sorted(pred, key= lambda x:x[2], reverse=True)
# topPredictionsLinks = [(k, v) for k,v,n in sortedPred]
predictions = sortedPred[0:k]
print(predictions[k-1])

(19, 1060, 1.2352458628142324)


In [69]:
#First y_true, y_pred

y_true = list()
y_pred = list()
thresholds = list()

for a,b,c in predictions:
    if (a,b) in set(testEdges) or (b,a) in set(testEdges):
        y_true.append(1)
        y_pred.append(c)
        print("There's a positive")
    else:
        y_true.append(0)
        y_pred.append(c)
    thresholds.append(c)





There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive
There's a positive


In [ ]:
def getAUPR():
    #Calculate the precision and recall at each threshold
    Recall = list()
    Precision =list()
    for index in range(1, len(predictions)+1):
        values = predictions[0:index]
        topPredictionsLinks = [(k, v) for k,v,n in values]
        # topPredictionsLinksReversed = [(v, k) for k,v,n in values]
        # topPredictionsLinks.extend(topPredictionsLinksReversed)

        TP = len([k for k in y_true[0:index] if k == 1])
        FP = len([k for k in y_true[0:index] if k == 0])

        FN = len(list(set(testEdges).difference(set(topPredictionsLinks))))

        Recall.append((TP)/((TP)+(FN)))
        Precision.append((TP)/((TP)+(FP)))
    print(f"The array lengths are thresholds {len(thresholds)}, recall {len(Recall)}, precision {len(Precision)}")
    df = pd.DataFrame({"Thresholds":thresholds,"Recall":Recall, "Precision":Precision})
    df = df.T
    display(df)

    aupr_score = auc(Recall, Precision)
    print(f"The aupr_score for Threshold K = {k} is {aupr_score}")
    #
    # plt.plot(Recall, Precision, label=f'AUPR = {aupr_score:.2f}')
    # plt.xlabel('Recall')
    # plt.ylabel('Precision')
    # plt.title('Precision-Recall Curve for Link Prediction')
    # plt.legend()
    # plt.show()
